In [1]:
import numpy as np
from scipy import stats
from math import ceil
from pathlib import Path
import matplotlib.pyplot as plt
from contact_plots import get_data_from_files, get_cdf_of_data, do_chi_square_test
import warnings
warnings.filterwarnings('ignore')

In [2]:
data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211215')

contact_data_list, infra_and_measure_name_2_data = get_data_from_files(data_folder)

In [3]:
data_names_small = (
    '1.0_customer_customer_weight_over_contacts',
    '1.0_customer_customer_weight_over_agents',
    '1.0_customer_customer_contacts_over_agents',
    '1.0_staff_customer_weight_over_contacts',
    '1.0_staff_customer_weight_over_agents',
    '1.0_staff_customer_contacts_over_agents',
    None,
    '1.0_customer_staff_weight_over_agents',
    '1.0_customer_staff_contacts_over_agents',
)

data_names_middle = (
    '1.5_customer_customer_weight_over_contacts',
    '1.5_customer_customer_weight_over_agents',
    '1.5_customer_customer_contacts_over_agents',
    '1.5_staff_customer_weight_over_contacts',
    '1.5_staff_customer_weight_over_agents',
    '1.5_staff_customer_contacts_over_agents',
    None,
    '1.5_customer_staff_weight_over_agents',
    '1.5_customer_staff_contacts_over_agents',
)

data_names_large = (
    '2.0_customer_customer_weight_over_contacts',    
    '2.0_customer_customer_weight_over_agents',
    '2.0_customer_customer_contacts_over_agents',
    '2.0_staff_customer_weight_over_contacts',
    '2.0_staff_customer_weight_over_agents',
    '2.0_staff_customer_contacts_over_agents',
    None,
    '2.0_customer_staff_weight_over_agents',
    '2.0_customer_staff_contacts_over_agents',
)

data_names_short = {
    '1.5_customer_customer_weight_over_contacts': 'cc_w_c',
    '1.5_customer_customer_weight_over_agents': 'cc_w_a',
    '1.5_customer_customer_contacts_over_agents': 'cc_c_a',
    '1.5_staff_customer_weight_over_contacts': 'sc_w_c',
    '1.5_staff_customer_weight_over_agents': 'sc_w_a',
    '1.5_staff_customer_contacts_over_agents': 'sc_c_a',
    '1.5_customer_staff_weight_over_agents': 'cs_w_a',
    '1.5_customer_staff_contacts_over_agents': 'cs_c_a',    

    '1.0_customer_customer_weight_over_contacts': 'cc_w_c',
    '1.0_customer_customer_weight_over_agents': 'cc_w_a',
    '1.0_customer_customer_contacts_over_agents': 'cc_c_a',
    '1.0_staff_customer_weight_over_contacts': 'sc_w_c',
    '1.0_staff_customer_weight_over_agents': 'sc_w_a',
    '1.0_staff_customer_contacts_over_agents': 'sc_c_a',
    '1.0_customer_staff_weight_over_agents': 'cs_w_a',
    '1.0_customer_staff_contacts_over_agents': 'cs_c_a',    

    '2.0_customer_customer_weight_over_contacts': 'cc_w_c',
    '2.0_customer_customer_weight_over_agents': 'cc_w_a',
    '2.0_customer_customer_contacts_over_agents': 'cc_c_a',
    '2.0_staff_customer_weight_over_contacts': 'sc_w_c',
    '2.0_staff_customer_weight_over_agents': 'sc_w_a',
    '2.0_staff_customer_contacts_over_agents': 'sc_c_a',
    '2.0_customer_staff_weight_over_agents': 'cs_w_a',
    '2.0_customer_staff_contacts_over_agents': 'cs_c_a',    
}

data_name_groups = {data_names_middle:'1.5',
                    data_names_small:'1.0',                      
                    data_names_large:'2.0'}

data_name_groups_inv = {value:key for key, value in data_name_groups.items()}

max_string_length = 0
for data_names in data_name_groups.keys():
    for data_name in data_names:
        if data_name is not None and len(data_name) > max_string_length:
            max_string_length = len(data_name)

In [4]:
lines = {
    'base_case':{'color':'r', 'linewidth':3, 'linestyle':'-'},
    'busy_corner':{'color':'b', 'linewidth':2, 'linestyle':'--'},
    'long_restaurant':{'color':'g', 'linewidth':2, 'linestyle':':'}    
}

base = 'nn_nts_bvd'

other_cases = [
    'n_nts_bvd',
    'nn_ts_bvd',
    'nn_nts_svd',
    'nn_nts_lvd'
    
]

In [5]:
%matplotlib wx



for ii in range(2):
    fig, ax = plt.subplots(1, 1, sharex=True)
    if ii == 0:
        stat_type = 'Mean'
    else:
        stat_type = 'Std'
        
    fig.suptitle(f'Relative differences - {stat_type}', fontsize=16)
    fig.canvas.manager.set_window_title(f'Relative differences - {stat_type}')

    x = []
    x_labels = []
    fill_x = True
    x_ind = 0
    y_data = {}
    ax.axhline(0, linewidth=2, color='k')
    first_infra_name = ''
    base_interaction_type = ''
    x_ranges_it = []
    it_labels = []
    x_ranges_dt = []
    dt_labels = []
    for infra_name, line_kwargs in lines.items():
        if len(first_infra_name) == 0:
            first_infra_name = infra_name
        
        y = []
        for data_name in data_names_middle:
            if data_name is None:
                continue
            
            if len(base_interaction_type) == 0:
                base_interaction_type = data_names_short[data_name].split('_')[0]
                it_labels.append(data_names_short[data_name].split('_')[0])
                x_ranges_it.append(0)
            
            if first_infra_name == infra_name:
                if data_names_short[data_name].split('_')[0] != base_interaction_type:
                    base_interaction_type = data_names_short[data_name].split('_')[0]
                    vline_color = '0.2'
                    x_ranges_it.append(x_ind)
                    it_labels.append(data_names_short[data_name].split('_')[0])
                else:
                    vline_color = '0.6'
                
                x_ranges_dt.append(x_ind)
                dt_labels.append('_'.join(data_names_short[data_name].split('_')[1:3]))
                ax.axvline(x_ind - 0.5, linewidth=2, color=vline_color, linestyle='--')
                
            base_contact_data = infra_and_measure_name_2_data[infra_name][base][data_name]
            
            data_names_short[data_name]
            if ii == 0:
                base_mean = np.mean(base_contact_data)
            else:
                base_var = np.std(base_contact_data)
            for other_case in other_cases:
                other_contact_data = infra_and_measure_name_2_data[infra_name][other_case][data_name]
                if ii == 0:
                    other_mean = np.mean(other_contact_data)
                    if fill_x:
                        x_labels.append(f'{data_names_short[data_name]} - {other_case}')
                        x.append(x_ind)
                        x_ind +=1
                    y.append(other_mean/base_mean - 1)
                else:
                    other_var = np.std(other_contact_data)
                    if fill_x:
                        x_labels.append(f'{data_names_short[data_name]} - {other_case}')
                        x.append(x_ind)
                        x_ind +=1
                    y.append(other_var/base_var - 1)

        fill_x = False
        ax.plot(x, y, label=f'{infra_name}', **line_kwargs)
    ylim = ax.get_ylim()
    ylim_diff = ylim[1] - ylim[0]
    for jj, x_coord_0 in enumerate(x_ranges_dt):
        if jj == len(x_ranges_dt) - 1:
            x_coord_1 = len(other_cases)*(len(data_names_middle) - 1) - 1
        else:
            x_coord_1 = x_ranges_dt[jj+1]
        
        ax.text(x_coord_0 + (x_coord_1 - x_coord_0)/2 - 0.5, ylim[1] - ylim_diff*0.04, dt_labels[jj], fontsize=18)
    
    print(x_ranges_it)
    print(it_labels)
    
    for jj, x_coord_0 in enumerate(x_ranges_it):
        if jj == len(x_ranges_it) - 1:
            x_coord_1 = len(other_cases)*(len(data_names_middle) - 1) - 1
        else:
            x_coord_1 = x_ranges_it[jj+1]
        
        ax.text(x_coord_0 + (x_coord_1 - x_coord_0)/2 - 0.5, ylim[1] + ylim_diff*0.04, it_labels[jj], fontsize=22)
    
    ax.set_ylim([ylim[0], ylim[1] + ylim_diff*0.10])
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, {'rotation':45, 'ha':'right', 'fontsize':12})
    ax.set_ylabel('Relative difference compared to base case')
    ax.grid()
    ax.legend()

    fig.subplots_adjust(bottom=0.16, wspace=0, hspace=0)
    
plt.show()


[0, 12, 24]
['cc', 'sc', 'cs']


TypeError: _set_ticklabels() takes 2 positional arguments but 3 were given